# Held-out DAS-only adjudication dashboard: what do the candidates mean?

This is a presentation and exploration dashboard for the current held-out adjudication checkpoint. It reads only compact, frozen or post-release adjudication products. It does not rerun detection, alter thresholds, delete candidates, query new catalogs, or assign repeater families.

The key distinction is deliberate: the figures show **DAS detection evidence** and **which independent checks have or have not succeeded**. They do not label the 21 candidates as earthquakes.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from IPython.display import display, Markdown

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "config" / "heldout_das_adjudication.json").is_file())
OUT = ROOT / "outputs" / "heldout_v2" / "adjudication"
partial = pd.read_csv(OUT / "partial_adjudication.csv")
generic = pd.read_csv(OUT / "forced_generic_network_scores.csv")
template = pd.read_csv(OUT / "forced_template_network_scores.csv")
wave = pd.read_csv(OUT / "das_only_waveform_review.csv")
config = json.loads((ROOT / "config" / "heldout_das_adjudication.json").read_text())

review = partial.merge(generic[["DAS_candidate_id", "generic_network_score_at_DAS_time", "generic_network_threshold"]], on="DAS_candidate_id")
review = review.merge(template[["DAS_candidate_id", "template_bank_score_at_DAS_time", "template_network_threshold"]], on="DAS_candidate_id")
review = review.merge(wave[["DAS_candidate_id", "DAS_v1_interval_null_threshold", "recomputed_score_max_pm_1s", "score_above_frozen_threshold_duration_pm_1s", "strong_block_count_at_candidate", "strong_block_count_max_pm_1s", "strong_block_support_duration_pm_1s", "coverage_status", "manual_waveform_review_status"]], on="DAS_candidate_id")
review["DAS_score_ratio"] = review["DAS_coincidence_score"] / review["DAS_v1_interval_null_threshold"]
review["generic_score_ratio"] = review["generic_network_score_at_DAS_time"] / review["generic_network_threshold"]
review["template_score_ratio"] = review["template_bank_score_at_DAS_time"] / review["template_network_threshold"]
review = review.sort_values(["DAS_score_ratio", "DAS_candidate_id"], ascending=[False, True]).reset_index(drop=True)
assert len(review) == 21
assert review["repeater_family_assignment"].eq("not_assigned").all()
display(review[["DAS_candidate_id", "interval_id", "DAS_score_ratio", "strong_block_count_at_candidate", "generic_score_ratio", "template_score_ratio", "cached_regional_catalog_status"]].head())


In [ ]:
plt.rcParams.update({"font.size": 10, "axes.titlesize": 12, "axes.labelsize": 10})
fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
fig.suptitle("SAFOD held-out DAS-only adjudication: evidence at a glance", fontsize=16, fontweight="bold")

# Decision funnel: these are checks, not event labels.
funnel_labels = ["DAS-only candidates", "Below generic network threshold", "Below template threshold", "No cached regional match", "Family assignments"]
funnel_values = [len(review), int((~review.generic_network_above_frozen_threshold).sum()), int((~review.template_network_above_frozen_threshold).sum()), int((review.cached_regional_catalog_status == "NO_CACHED_BROAD_REGIONAL_EVENT_WITHIN_30S").sum()), 0]
colors = ["#35618f", "#4f83b5", "#6da5c9", "#94c6d9", "#d9e7ed"]
axes[0,0].barh(funnel_labels[::-1], funnel_values[::-1], color=colors[::-1])
for y, value in enumerate(funnel_values[::-1]): axes[0,0].text(value + 0.3, y, str(value), va="center", fontweight="bold")
axes[0,0].set_xlim(0, 23); axes[0,0].set_title("Independent checks (not classifications)"); axes[0,0].set_xlabel("candidate count")

# Ranked DAS support.
ranked = review.sort_values("DAS_score_ratio", ascending=True)
axes[0,1].scatter(ranked["DAS_score_ratio"], np.arange(len(ranked)), c=ranked["strong_block_count_at_candidate"], cmap="viridis", s=65, edgecolor="white", linewidth=0.5)
axes[0,1].axvline(1.0, color="black", linestyle="--", lw=1, label="frozen DAS threshold")
axes[0,1].set_yticks(np.arange(len(ranked))); axes[0,1].set_yticklabels(ranked["DAS_candidate_id"], fontsize=7)
axes[0,1].set_xlabel("DAS score / interval threshold"); axes[0,1].set_title("DAS candidate strength and block support")
axes[0,1].legend(fontsize=8, loc="lower right")

# Evidence matrix.
evidence = pd.DataFrame({
    "DAS window coverage": review["coverage_status"].eq("PASS").astype(int),
    "4+ strong DAS blocks": (review["strong_block_count_at_candidate"] >= 4).astype(int),
    "generic network crossing": review["generic_network_above_frozen_threshold"].astype(int),
    "template crossing": review["template_network_above_frozen_threshold"].astype(int),
    "regional catalog match": (review["cached_regional_catalog_status"] != "NO_CACHED_BROAD_REGIONAL_EVENT_WITHIN_30S").astype(int),
}).T
axes[1,0].imshow(evidence.values, aspect="auto", cmap=ListedColormap(["#edf3f5", "#2d7f72"]), vmin=0, vmax=1)
axes[1,0].set_yticks(range(len(evidence.index))); axes[1,0].set_yticklabels(evidence.index, fontsize=9)
axes[1,0].set_xticks(range(len(review))); axes[1,0].set_xticklabels([str(i+1) for i in range(len(review))], fontsize=7)
axes[1,0].set_xlabel("candidate rank by DAS score"); axes[1,0].set_title("Evidence matrix: green means check passed")

# Interval stratification.
interval_summary = review.groupby("interval_id").agg(candidates=("DAS_candidate_id", "size"), median_score_ratio=("DAS_score_ratio", "median"), median_support_duration_s=("strong_block_support_duration_pm_1s", "median")).reset_index()
x = np.arange(len(interval_summary))
axes[1,1].bar(x - 0.18, interval_summary["candidates"], width=0.36, label="candidates", color="#35618f")
axes2 = axes[1,1].twinx(); axes2.plot(x + 0.18, interval_summary["median_score_ratio"], "o-", color="#d97732", label="median DAS ratio")
axes[1,1].set_xticks(x); axes[1,1].set_xticklabels(interval_summary["interval_id"]); axes[1,1].set_ylabel("candidate count"); axes2.set_ylabel("median DAS score / threshold")
axes[1,1].set_title("Interval-stratified candidate burden")
fig.savefig(OUT / "heldout_das_adjudication_overview.png", dpi=180, bbox_inches="tight")
display(fig)


In [ ]:
def show_candidate(candidate_number=1):
    row = review.iloc[int(candidate_number) - 1]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    fig.suptitle(f"Candidate {candidate_number}: {row.DAS_candidate_id} ({row.interval_id})", fontsize=14, fontweight="bold")
    labels = ["DAS score ratio", "generic network ratio", "template network ratio"]
    values = [row.DAS_score_ratio, row.generic_score_ratio, row.template_score_ratio]
    bars = axes[0].barh(labels[::-1], values[::-1], color=["#35618f", "#8aaec4", "#b7cbd5"][::-1])
    axes[0].axvline(1.0, color="black", linestyle="--", lw=1)
    axes[0].set_xlim(0, max(1.2, max(values) * 1.15)); axes[0].set_xlabel("score / frozen threshold"); axes[0].set_title("Threshold-relative scores")
    for bar, value in zip(bars, values[::-1]): axes[0].text(value + 0.02, bar.get_y() + bar.get_height()/2, f"{value:.2f}", va="center")
    labels2 = ["strong blocks", "max blocks ±1 s", "support duration (s)"]
    values2 = [row.strong_block_count_at_candidate, row.strong_block_count_max_pm_1s, row.strong_block_support_duration_pm_1s]
    axes[1].bar(labels2, values2, color=["#2d7f72", "#5ba99a", "#9acfc3"]); axes[1].set_title("DAS persistence/support"); axes[1].tick_params(axis="x", rotation=25)
    axes[1].set_ylim(0, max(1, max(values2) * 1.25))
    display(fig)
    plt.close(fig)
    display(Markdown(f"**Status:** DAS coverage `{row.coverage_status}`; generic crossing `{row.generic_network_above_frozen_threshold}`; template crossing `{row.template_network_above_frozen_threshold}`; cached regional match `{row.cached_regional_catalog_status}`; manual waveform review `{row.manual_waveform_review_status}`.\n\n**Interpretation:** this is a candidate-evidence view, not an earthquake or repeater-family decision."))

# Change this number and rerun the cell to explore candidates.
candidate_number = 1
show_candidate(candidate_number)


In [ ]:
controls = config["known_positive_controls"]["controls"]
print("Development positive controls (kept separate from held-out candidates):")
display(pd.DataFrame(controls)[["DAS_candidate_id", "network_event_id", "DAS_score_rank", "strong_block_count_of_10"]])
print("2/2 known local development events were recovered; this is a capability sanity check, not a held-out recall estimate.")


## How to use this dashboard

- Start with the overview: it separates **DAS detection evidence** from independent checks.
- Change `candidate_number` in the candidate-browser cell and rerun it to inspect any row.
- Use the score ratios and support-duration bars to prioritize waveform review; do not treat a high ratio as proof of an earthquake.
- The next scientific figure should add conventional station traces for selected candidates. The current dashboard intentionally labels those checks as pending rather than implying that the network “miss” is a validated extension.
